# For Various Operations Needed

## Create the folders needed for generating dataset.

### For Training Dataset

In [4]:
import numpy as np
import os

PATH = "/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_DJISpark/drop_region/data/"
# PATH = "/home/wlau0003/Reuben_ws/FANET_Dataset/Dataset_NP100000_DJISpark/train_dataset_mar25/"
# PATH = "/home/clow0003/Reuben_ws/FANET_Dataset/Dataset_NP100000_DJISpark/train_dataset_mar25/"
# bitrates = [6.5, 13, 19.5, 26, 39, 52, 58.5, 65]
bitrates = [39]
heights = np.arange(60, 330, 30)
usis = [10, 20, 66.7, 100]

for bitrate in bitrates:
    for height in heights:
        for usi in usis:
            os.mkdir(os.path.join(PATH, f"BitRate-{bitrate}_Height-{height}_UAVSendingInterval-{usi}"))

### For Testing Dataset

In [9]:
import numpy as np
import os

PATH = "/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_ParrotAR2/test_dataset/"
bitrates = [6.5, 6.5, 6.5, 6.5, 13, 13, 13, 13, 19.5, 19.5, 19.5, 19.5, 26, 26, 26, 26,
            39, 39, 39, 39, 52, 52, 52, 52, 58.5, 58.5, 58.5, 58.5, 65, 65, 65, 65,
            65, 65, 65, 65, 58.5, 58.5, 58.5, 58.5, 52, 52, 52, 52, 39, 39, 39, 39, 26, 26, 26, 26, 19.5, 19.5, 19.5, 19.5, \
            13, 13, 13, 13, 6.5, 6.5, 6.5, 6.5]
heights = [75, 135, 195, 255, 75, 135, 195, 255, 75, 135, 195, 255, 75, 135, 195, 255,
            105, 165, 225, 285, 105, 165, 225, 285, 105, 165, 225, 285, 105, 165, 225, 285,
            75, 135, 195, 255, 75, 135, 195, 255, 75, 135, 195, 255, 75, 135, 195, 255,
            105, 165, 225, 285, 105, 165, 225, 285, 105, 165, 225, 285, 105, 165, 225, 285]
usis = [10, 20, 66.7, 100, 20, 10, 100, 66.7, 66.7, 100, 10, 20, 100, 66.7, 20, 10,
        10, 20, 66.7, 100, 20, 10, 100, 66.7, 66.7, 100, 10, 20, 100, 66.7, 20, 10,
        10, 20, 66.7, 100, 20, 10, 100, 66.7, 66.7, 100, 10, 20, 100, 66.7, 20, 10,
        10, 20, 66.7, 100, 20, 10, 100, 66.7, 66.7, 100, 10, 20, 100, 66.7, 20, 10]

for bitrate, height, usi in zip(bitrates, heights, usis):
    os.mkdir(os.path.join(PATH, f"BitRate-{bitrate}_Height-{height}_UAVSendingInterval-{usi}"))


## Check That Processed Data Has At Least NPR Packets

In [13]:
import pandas as pd
import glob
import os
PATH = "/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_DJIMavicAir/train_dataset_processed_39_52/"
files = glob.glob(os.path.join(PATH, "*.csv"))
for file in files:
    df = pd.read_csv(file)
    if "Num_Fail" in df.columns:
        df.drop(columns=["Num_Fail"], inplace=True)
        df.to_csv(file, index=False)
    df["Num_Packets"] = df["Num_Reliable"] + df["Num_Delay_Excd"] + df["Num_Fail_Other"]
    print(len(df.loc[df["Num_Packets"] < 100000]))


0
0
0
0
0
0
0
0
0
0


## Concatenate Processed Datasets

In [7]:
import pandas as pd
import os

PATHS = ["/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_DJIMavicAir/train_dataset_processed_6.5_13_19.5_26",
         "/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_DJIMavicAir/train_dataset_processed_39_52",
         "/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_DJIMavicAir/train_dataset_processed_58.5_65"]
FILES = ["Uplink_Reliability.csv", "Video_Reliability.csv", "Downlink_GW_Reliability.csv", "Downlink_UAV-0_Reliability.csv", "Downlink_UAV-1_Reliability.csv", "Downlink_UAV-2_Reliability.csv",
         "Downlink_UAV-3_Reliability.csv", "Downlink_UAV-4_Reliability.csv", "Downlink_UAV-5_Reliability.csv", "Downlink_UAV-6_Reliability.csv"]
SAVE_PATH = "/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_DJIMavicAir/train_dataset_processed"

# Concat each file in FILES from each path in PATHS
for file in FILES:
    df_list = []
    for path in PATHS:
        df_list.append(pd.read_csv(os.path.join(path, file)))
    df = pd.concat(df_list)
    df.to_csv(os.path.join(SAVE_PATH, file), index=False)

# For Exploring Dataset

In [ ]:
# Find max abs difference in simulated reliability for each UAV
import pandas as pd
import os
import numpy as np

PATH = "/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_DJISpark/train_dataset_processed"
FILES = ["Uplink_GW_Reliability.csv", "Uplink_UAV-0_Reliability.csv", "Uplink_UAV-1_Reliability.csv", "Uplink_UAV-2_Reliability.csv",
         "Uplink_UAV-3_Reliability.csv", "Uplink_UAV-4_Reliability.csv", "Uplink_UAV-5_Reliability.csv", "Uplink_UAV-6_Reliability.csv"]

for i in range(len(FILES)):
    for j in range(i + 1, len(FILES)):
        file1 = FILES[i]
        file2 = FILES[j]
        # Read the two files
        df1 = pd.read_csv(os.path.join(PATH, file1))
        df2 = pd.read_csv(os.path.join(PATH, file2))
        df = pd.DataFrame()
        # Get Num_Fail column
        df1["Num_Fail"] = df1["Num_Delay_Excd"] + df1["Num_Fail_Other"] 
        df2["Num_Fail"] = df2["Num_Delay_Excd"] + df2["Num_Fail_Other"]
        # Find the percent difference between the Num_Reliable and Num_Fail columns
        # NOTE: Difference for Num_Reliable and Num_Fail columns should be the same
        df["Difference_Num_Reliable"] = np.abs(df1["Num_Reliable"] - df2["Num_Reliable"]) 
        df["Difference_Num_Fail"] = np.abs(df1["Num_Fail"] - df2["Num_Fail"]) 
        # df["Total_Difference"] = df["Difference_Num_Reliable"] + df["Difference_Num_Fail"]
        # Find the max percent difference for each UAV
        max_difference = np.max([df["Difference_Num_Reliable"].max(), df["Difference_Num_Fail"].max()])
        # max_difference = df["Total_Difference"].max()
        print(f"Max percent difference for {file1} and {file2}: {max_difference}")

Max percent difference for Uplink_GW_Reliability.csv and Uplink_UAV-0_Reliability.csv: 458
Max percent difference for Uplink_GW_Reliability.csv and Uplink_UAV-1_Reliability.csv: 310
Max percent difference for Uplink_GW_Reliability.csv and Uplink_UAV-2_Reliability.csv: 236
Max percent difference for Uplink_GW_Reliability.csv and Uplink_UAV-3_Reliability.csv: 295
Max percent difference for Uplink_GW_Reliability.csv and Uplink_UAV-4_Reliability.csv: 282
Max percent difference for Uplink_GW_Reliability.csv and Uplink_UAV-5_Reliability.csv: 234


Max percent difference for Uplink_GW_Reliability.csv and Uplink_UAV-6_Reliability.csv: 333
Max percent difference for Uplink_UAV-0_Reliability.csv and Uplink_UAV-1_Reliability.csv: 237
Max percent difference for Uplink_UAV-0_Reliability.csv and Uplink_UAV-2_Reliability.csv: 464
Max percent difference for Uplink_UAV-0_Reliability.csv and Uplink_UAV-3_Reliability.csv: 586
Max percent difference for Uplink_UAV-0_Reliability.csv and Uplink_UAV-4_Reliability.csv: 603
Max percent difference for Uplink_UAV-0_Reliability.csv and Uplink_UAV-5_Reliability.csv: 502
Max percent difference for Uplink_UAV-0_Reliability.csv and Uplink_UAV-6_Reliability.csv: 251
Max percent difference for Uplink_UAV-1_Reliability.csv and Uplink_UAV-2_Reliability.csv: 399
Max percent difference for Uplink_UAV-1_Reliability.csv and Uplink_UAV-3_Reliability.csv: 495
Max percent difference for Uplink_UAV-1_Reliability.csv and Uplink_UAV-4_Reliability.csv: 526
Max percent difference for Uplink_UAV-1_Reliability.csv and Upl

In [4]:
import pandas as pd

df = pd.read_csv("/media/research-student/DataDrive/FANET_Dataset/Dataset_NP100000_DJISpark/temp/NumMember-7_BitRate-13_Height-60_Distance-0_Modulation-QPSK_UAVSendingInterval-10/GCS-Rx.csv")
df["Delay"] = df["RxTime"] - df["TxTime"]
len(df.loc[df["Delay"] < 0.04])/len(df)

0.004714661239718477